# Infinite-Width Kernel Predictions on Histological Data

This notebook compares a standard CNN against a C4-invariant CNN using
infinite-width NTK kernel regression on the NCT-CRC-HE histological image dataset
(9 tissue classes).

All layers before the global pooling operation are equivariant.

## Preparation: Data Download

The cell below downloads the NCT-CRC-HE-100K dataset (~10 GB) from [Zenodo](https://zenodo.org/records/1214456) the first
time it is run.  Set `DATA_DIR` to any location with sufficient space; subsequent runs
skip the download if the directory already exist

In [3]:
from pathlib import Path

DATA_DIR = Path('data')

In [4]:
import urllib.request
import zipfile

DATASET_PATH = DATA_DIR / 'NCT-CRC-HE-100K'

if DATASET_PATH.exists():
    print(f'Dataset found at {DATASET_PATH}')
else:
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    zip_path = DATA_DIR / 'NCT-CRC-HE-100K.zip'

    if not zip_path.exists():
        url = 'https://zenodo.org/records/1214456/files/NCT-CRC-HE-100K.zip?download=1'
        print(f'Downloading NCT-CRC-HE-100K (~10 GB) to {zip_path} ...')

        def _progress(block_num, block_size, total_size):
            if total_size > 0:
                pct = min(block_num * block_size / total_size * 100, 100)
                print(f'\r  {pct:.1f}%', end='', flush=True)

        urllib.request.urlretrieve(url, zip_path, reporthook=_progress)
        print()

    print('Extracting ...')
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(DATA_DIR)
    print(f'Done. Dataset at {DATASET_PATH}')

Dataset found at data/NCT-CRC-HE-100K


## Setup

In [5]:
import jax
import jax.numpy as jnp
from jax import jit
import neural_tangents as nt
from neural_tangents import stax
from neural_tangents.stax import Padding

from equivariant_ntk.layers import GroupPool, P4ConvP4, Z2ConvP4
from dataset import create_dataset_for_jax, create_dataset_tensors

print(f'JAX devices: {jax.devices()}')

JAX devices: [CpuDevice(id=0)]


### Data

In [6]:
SEED     = 343046395
N_TRAIN  = 8
N_TEST   = 100
PIXELS   = 32   # resize images to PIXELS×PIXELS; reduce for faster CPU runs

dataset_raw = create_dataset_for_jax(str(DATASET_PATH), SEED, PIXELS, N_TRAIN, N_TEST)
data = create_dataset_tensors(*dataset_raw[:2])

print(f'Train: {data.train_data.shape}  labels: {data.train_labels.shape}')
print(f'Test:  {data.test_data.shape}   labels: {data.test_labels.shape}')

Loaded dataset
 NCT-CRC-HE dataset with the following classes:
 0: ADI  consisting of 10407
 1: BACK consisting of 10566
 2: DEB  consisting of 11512
 3: LYM  consisting of 11557
 4: MUC  consisting of  8896
 5: MUS  consisting of 13536
 6: NORM consisting of  8763
 7: STR  consisting of 10446
 8: TUM  consisting of 14317

Train: (8, 32, 32, 3)  labels: (8, 9)
Test:  (100, 32, 32, 3)   labels: (100, 9)


### Models

Both models share the same local 3×3 filter structure and the same output dimension. For this purely infinite-width experiment, the channel widths do not matter but they need to be specified.

In [7]:
def make_cnn():
    """Standard CNN (equivariant only to translations)."""
    return stax.serial(
        stax.Conv(64, (3, 3), padding='VALID'),
        stax.Relu(),
        stax.Conv(64, (3, 3), padding='VALID'),
        stax.Relu(),
        stax.GlobalAvgPool(),
        stax.Dense(32),
        stax.Relu(),
        stax.Dense(9),
    )


def make_p4cnn():
    """C4-invariant CNN (equivariant to translations and 90° rotations)."""
    return stax.serial(
        Z2ConvP4(64, (3, 3), padding=Padding.VALID),
        stax.Relu(),
        P4ConvP4(64, (3, 3), padding=Padding.VALID),
        stax.Relu(),
        GroupPool(),
        stax.Dense(32),
        stax.Relu(),
        stax.Dense(9),
    )

## Kernel Regression

We use `nt.predict.gradient_descent_mse_ensemble` which computes the analytic
prediction of gradient-flow training at $t \to \infty$, in the infinite-width limit. Since this method assumes an MSE loss, we transform the classification problem into a regression problem. A small diagonal regulariser `DIAG_REG` stabilises the kernel inversion. This becomes relevant for larger training sets.

In [8]:
BATCH_SIZE = 4    # kernel computation batch size; reduce if memory is tight
DIAG_REG   = 1e-4  # diagonal regularisation for kernel inversion


def run_kernel_prediction(kernel_fn, data):
    """Compute test accuracy using infinite-width NTK kernel regression."""
    # nt.batch also applies jit internally to kernel_fn
    kernel_fn_batched = nt.batch(kernel_fn, batch_size=BATCH_SIZE)

    # Centre one-hot labels: subtract uniform prior 1/<num classes>
    train_labels_centered = data.train_labels - 1 / 9

    predict_fn = nt.predict.gradient_descent_mse_ensemble(
        kernel_fn_batched,
        data.train_data,
        train_labels_centered,
        diag_reg=DIAG_REG,
    )

    preds = predict_fn(x_test=data.test_data, get='ntk', compute_cov=False)
    max_prints = min(15, preds.shape[0])
    for pred, label in zip(preds[:max_prints], data.test_labels[:max_prints]):
        pred = jnp.argmax(pred, axis=-1)
        label = jnp.argmax(label, axis=-1)
        suffix = "✓" if pred == label else ""
        print(f"pred: {pred}, label: {label}  {suffix}")  
    if preds.shape[0] > max_prints:
        print(f"... (total {preds.shape[0]} predictions)")

    test_labels_int = jnp.argmax(data.test_labels, axis=1)
    acc = float(jnp.mean(jnp.argmax(preds, axis=1) == test_labels_int))
    return acc

## CNN

In [9]:
print('Running CNN kernel regression ...')
_, _, cnn_kernel_fn = make_cnn()
cnn_acc = run_kernel_prediction(cnn_kernel_fn, data)
print(f'CNN test accuracy: {cnn_acc:.1%}')

Running CNN kernel regression ...
pred: 5, label: 4  
pred: 4, label: 8  
pred: 7, label: 5  
pred: 8, label: 6  
pred: 0, label: 0  ✓
pred: 8, label: 3  
pred: 1, label: 1  ✓
pred: 8, label: 3  
pred: 8, label: 3  
pred: 8, label: 6  
pred: 7, label: 2  
pred: 7, label: 2  
pred: 0, label: 0  ✓
pred: 8, label: 8  ✓
pred: 8, label: 3  
... (total 100 predictions)
CNN test accuracy: 25.0%


## $C_4$-invariant CNN
Due to the extra group dimensions, the kernel gets two additional dimensions of size 4. This and additional operations, will make the following kernel regression noticably slower than the pure CNN.

In [10]:
print('Running P4CNN kernel regression ...')
_, _, p4cnn_kernel_fn = make_p4cnn()
p4cnn_acc = run_kernel_prediction(p4cnn_kernel_fn, data)
print(f'P4CNN test accuracy: {p4cnn_acc:.1%}')

Running P4CNN kernel regression ...
pred: 5, label: 4  
pred: 4, label: 8  
pred: 7, label: 5  
pred: 8, label: 6  
pred: 0, label: 0  ✓
pred: 8, label: 3  
pred: 1, label: 1  ✓
pred: 8, label: 3  
pred: 8, label: 3  
pred: 8, label: 6  
pred: 7, label: 2  
pred: 7, label: 2  
pred: 0, label: 0  ✓
pred: 8, label: 8  ✓
pred: 8, label: 3  
... (total 100 predictions)
P4CNN test accuracy: 28.0%


## Results

In [12]:
print(f'Infinite-width NTK prediction accuracy  ({N_TRAIN} train / {N_TEST} test, {PIXELS}×{PIXELS} px)')
print(f'{"":-<40}')
print(f'{"CNN":10s}  {cnn_acc:>8.1%}')
print(f'{"C4-CNN":10s}  {p4cnn_acc:>8.1%}')

Infinite-width NTK prediction accuracy  (8 train / 100 test, 32×32 px)
----------------------------------------
CNN            25.0%
C4-CNN         28.0%
